# FTMO AI Trading System — Cloud Inference
**Runs LightGBM + CatBoost inference on all 8 pretrained symbol pairs.**

### Setup required (one-time):
1. Go to **Colab Secrets** (🔑 icon in left sidebar)
2. Add secret named `AZURE_STORAGE_CONNECTION_STRING` with your full Azure connection string
3. Enable 'Notebook access' for the secret
4. Run all cells (Runtime → Run all)

The inference loop runs forever — leave this tab open during trading hours.

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install -q azure-storage-blob lightgbm catboost xgboost joblib pandas numpy

In [ ]:
# ── Cell 2: Clone repo to get models + feature_expander ─────────────────
import os
if not os.path.exists('/content/AI-Trading-System'):
    !git clone -q https://github.com/Codebreaker-source/AI-Trading-System.git /content/AI-Trading-System
else:
    !git -C /content/AI-Trading-System pull -q

import sys
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system')
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system/core')
print('Repo ready.')
!ls /content/AI-Trading-System/models/current/

In [ ]:
# ── Cell 3: Load Azure connection string from Colab Secrets ─────────────
from google.colab import userdata
import os

try:
    AZURE_CONN_STR = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
    os.environ['AZURE_STORAGE_CONNECTION_STRING'] = AZURE_CONN_STR
    print('Azure connection string loaded from Colab Secrets.')
except Exception as e:
    raise RuntimeError(
        'AZURE_STORAGE_CONNECTION_STRING not found in Colab Secrets. '
        'Add it via the key icon in the left sidebar.'
    ) from e

In [ ]:
# ── Cell 4: Load all LightGBM + CatBoost models ─────────────────────────
import joblib
import glob

MODEL_DIR = '/content/AI-Trading-System/models/current'

# 8 pretrained base symbols
PRETRAINED_BASES = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF',
                    'AUDUSD', 'USDCAD', 'NZDUSD', 'EURGBP']

lgbm_models     = {}
catboost_models = {}

for base in PRETRAINED_BASES:
    lgbm_path = f'{MODEL_DIR}/{base}_lightgbm.joblib'
    cb_path   = f'{MODEL_DIR}/{base}_catboost.joblib'

    if os.path.exists(lgbm_path):
        lgbm_models[base] = joblib.load(lgbm_path)
        print(f'  LGB loaded: {base}')
    else:
        print(f'  LGB MISSING: {lgbm_path}')

    if os.path.exists(cb_path):
        catboost_models[base] = joblib.load(cb_path)
        print(f'  CAT loaded: {base}')
    else:
        print(f'  CAT MISSING: {cb_path}')

print(f'\nLoaded: {len(lgbm_models)} LightGBM + {len(catboost_models)} CatBoost models')

In [ ]:
# ── Cell 5: Load feature expander ───────────────────────────────────────
from feature_expander import expand_features
import numpy as np
import pandas as pd
print('Feature expander loaded.')

In [ ]:
# ── Cell 6: Azure helpers ────────────────────────────────────────────────
from azure.storage.blob import BlobServiceClient
import json
from datetime import datetime, timezone
import io

FEATURES_CONTAINER    = 'trading-features'
PREDICTIONS_CONTAINER = 'colab-predictions'

blob_service = BlobServiceClient.from_connection_string(AZURE_CONN_STR)

# Ensure containers exist
for c in [FEATURES_CONTAINER, PREDICTIONS_CONTAINER]:
    try:
        blob_service.create_container(c)
        print(f'Created container: {c}')
    except:
        print(f'Container exists: {c}')

def download_features(symbol: str):
    """Download feature CSV for a symbol from Azure. Returns DataFrame or None."""
    try:
        blob = blob_service.get_blob_client(container=FEATURES_CONTAINER,
                                            blob=f'{symbol}_features.csv')
        raw = blob.download_blob().readall().decode('utf-8')
        df  = pd.read_csv(io.StringIO(raw))
        return df
    except Exception:
        return None

def upload_prediction(symbol: str, source: str, action: str, confidence: float):
    """Upload inference result to Azure."""
    payload = {
        'symbol':     symbol,
        'source':     source,
        'action':     action,
        'confidence': round(float(confidence), 4),
        'timestamp':  datetime.now(timezone.utc).isoformat(),
    }
    blob = blob_service.get_blob_client(container=PREDICTIONS_CONTAINER,
                                        blob=f'{symbol}_{source}_pred.json')
    blob.upload_blob(json.dumps(payload), overwrite=True)

print('Azure helpers ready.')

In [ ]:
# ── Cell 7: Prediction helpers (HOLD-bias fix) ───────────────────────────
#
# Label encoding: 0=SELL  1=HOLD  2=BUY
# LightGBM predicts HOLD ~100% on several pairs (class imbalance during training).
# Fix: if HOLD wins but a directional class exceeds BIAS_THRESHOLD, use it instead.

BIAS_THRESHOLD = 0.35   # directional prob must exceed this to override HOLD

LABEL_MAP = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}

def predict_with_bias_fix(model, features_105: np.ndarray):
    """
    Run model.predict_proba(), apply HOLD-bias override.
    Returns (action: str, confidence: float)
    """
    try:
        proba    = model.predict_proba(features_105.reshape(1, -1))[0]
        sell_p   = float(proba[0])
        hold_p   = float(proba[1])
        buy_p    = float(proba[2])
        pred_idx = int(np.argmax(proba))

        if pred_idx == 1:   # HOLD wins — apply bias fix
            if buy_p >= BIAS_THRESHOLD and buy_p > sell_p:
                return 'BUY',  buy_p
            if sell_p >= BIAS_THRESHOLD and sell_p > buy_p:
                return 'SELL', sell_p
            return 'HOLD', hold_p

        return LABEL_MAP[pred_idx], float(proba[pred_idx])

    except Exception as e:
        print(f'predict error: {e}')
        return 'HOLD', 0.0


def run_inference_for_symbol(symbol_full: str, base: str):
    """
    Download features, expand 27→105, run LGB + CatBoost, upload predictions.
    symbol_full = 'EURUSD.sim'   base = 'EURUSD'
    """
    df = download_features(symbol_full)
    if df is None or df.empty:
        return None, None

    # Get latest row as 27-feature array
    latest = df.iloc[-1].values.astype(np.float32)
    if len(latest) < 27:
        return None, None
    features_27  = latest[:27]

    # Expand to 105 features
    try:
        features_105 = expand_features(features_27)
    except Exception as e:
        print(f'Feature expand failed {base}: {e}')
        return None, None

    lgbm_result = catboost_result = None

    # LightGBM
    if base in lgbm_models:
        action, conf = predict_with_bias_fix(lgbm_models[base], features_105)
        lgbm_result  = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'lgbm', action, conf)

    # CatBoost
    if base in catboost_models:
        action, conf   = predict_with_bias_fix(catboost_models[base], features_105)
        catboost_result = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'catboost', action, conf)

    return lgbm_result, catboost_result

print('Inference helpers ready.')

In [ ]:
# ── Cell 8: Symbol mapping (full OANDA name → base name) ─────────────────
# Maps OANDA .sim suffix names to the base names used for model loading.
# Add more entries here if OANDA uses different suffixes.

SYMBOL_MAP = {
    'EURUSD.sim': 'EURUSD',
    'GBPUSD.sim': 'GBPUSD',
    'USDJPY.sim': 'USDJPY',
    'USDCHF.sim': 'USDCHF',
    'AUDUSD.sim': 'AUDUSD',
    'USDCAD.sim': 'USDCAD',
    'NZDUSD.sim': 'NZDUSD',
    'EURGBP.sim': 'EURGBP',
}
print(f'Symbol map: {len(SYMBOL_MAP)} symbols configured')

In [ ]:
# ── Cell 9: MAIN INFERENCE LOOP — runs forever ───────────────────────────
# Leave this cell running. It polls Azure every 30 seconds,
# runs inference on all 8 symbols, and uploads predictions.
# Colab will keep the session alive as long as this cell is executing.

import time
from IPython.display import clear_output

POLL_INTERVAL = 30   # seconds between inference runs
cycle = 0

print('=== FTMO AI Trading System — Colab Inference ===')
print(f'Models: {len(lgbm_models)} LGB + {len(catboost_models)} CatBoost')
print(f'Symbols: {list(SYMBOL_MAP.keys())}')
print(f'Poll interval: {POLL_INTERVAL}s')
print('Running... (interrupt kernel to stop)')
print()

while True:
    cycle += 1
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    results_log = []

    for sym_full, base in SYMBOL_MAP.items():
        lgbm_r, cb_r = run_inference_for_symbol(sym_full, base)

        lgbm_str = f"{lgbm_r[0]} ({lgbm_r[1]:.2f})" if lgbm_r else 'no data'
        cb_str   = f"{cb_r[0]} ({cb_r[1]:.2f})"   if cb_r   else 'no data'
        results_log.append(f'  {base:<8}  LGB: {lgbm_str:<20}  CAT: {cb_str}')

    clear_output(wait=True)
    print(f'=== Cycle {cycle} · {ts} ===')
    for line in results_log:
        print(line)
    print(f'\nNext run in {POLL_INTERVAL}s...')

    time.sleep(POLL_INTERVAL)